## GRPO Reimplementation

In [2]:
import os
import re
import torch
import weave
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl.trainer import GRPOConfig, GRPOTrainer
from tqdm import tqdm
import json
from datetime import datetime

# Prompt / XML formatting
SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

Initial reward functions

In [ ]:
def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

# Dataset preparation
def get_gsm8k_questions(split="train"):
    data = load_dataset("openai/gsm8k", "main")[split]
    def map_fn(x):
        return {
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": x["question"]}
            ],
            "answer": extract_hash_answer(x["answer"])
        }
    data = data.map(map_fn)
    return data

dataset = get_gsm8k_questions()
print("Dataset ready:", len(dataset))

# Reward functions
def correctness_reward_func(prompts, completions, answer, **kwargs):
    responses = [c[0]['content'] for c in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def int_reward_func(completions, **kwargs):
    responses = [c[0]['content'] for c in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

def soft_format_reward_func(completions, **kwargs):
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [c[0]['content'] for c in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

# Load Qwen 0.5B model 
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map={"": 0},      
    max_memory={0: "35GB"},   # use 35 GB for model, leave 5GB buffer
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# GRPO configuration
output_dir = "Qwen2-0.5B-Instruct-gsm8k-grpo"
training_args = GRPOConfig(
    per_device_train_batch_size=16,  
    gradient_accumulation_steps=4,   
    max_steps=1000,
    bf16=True,
    report_to="wandb",
    run_name="grpo-gsm8k-xml",
    logging_steps=10,
    max_prompt_length=512,
    max_completion_length=512,
)


# Initialize GRPO trainer
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[correctness_reward_func, int_reward_func, soft_format_reward_func],
    train_dataset=dataset,
    args=training_args,
)

# Start training
trainer.train()
trainer.save_model(output_dir)


Dataset ready: 7473


The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.002000
70,0.000000
80,0.000000
90,0.000000
100,0.002300


In [ ]:
# Setup & Constants
max_seq_length = 1024 
max_prompt_length = 256

# Data Preprocessing
def extract_hash_answer(text: str) -> str | None:
    if "####" not in text: return None
    return text.split("####")[1].strip()

def get_gsm8k_questions(split="train"):
    # Load a tiny subset for demonstration speed, remove slice for full training
    data = load_dataset("openai/gsm8k", "main", split=split)
    data = data.map(lambda x: {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": x["question"]}
        ],
        "answer": extract_hash_answer(x["answer"])
    })
    return data

dataset = get_gsm8k_questions()

In [8]:
def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def correctness_reward_func(prompts, completions, answer, **kwargs):
    responses = [c[0]['content'] for c in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def int_reward_func(completions, **kwargs):
    responses = [c[0]['content'] for c in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

def soft_format_reward_func(completions, **kwargs):
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [c[0]['content'] for c in completions]
    matches = [re.search(pattern, r, flags=re.DOTALL) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def strict_format_reward_func(completions, **kwargs):
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [c[0]["content"] for c in completions]
    matches = [re.search(pattern, r, flags=re.DOTALL) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def xmlcount_reward_func(completions, **kwargs):
    responses = [c[0]['content'] for c in completions]
    def count_tags(text):
        score = 0.0
        if "<reasoning>" in text: score += 0.125
        if "</reasoning>" in text: score += 0.125
        if "<answer>" in text: score += 0.125
        if "</answer>" in text: score += 0.125
        return score
    return [count_tags(r) for r in responses]

# Training with correct reward functions

In [ ]:
# Model & Trainer Setup
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
output_dir = "outputs"

training_args = GRPOConfig(
    output_dir=output_dir,
    learning_rate=5e-6,
    logging_steps=10,
    per_device_train_batch_size=16, 
    gradient_accumulation_steps=4, 
    num_generations=4,              
    max_prompt_length=512,
    max_completion_length=512,
    max_steps=500,
    save_steps=250,
    run_name="grpo-gsm8k-xml-fast",
    report_to="wandb",  
    bf16=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        xmlcount_reward_func,
        soft_format_reward_func,
        strict_format_reward_func,
        int_reward_func,
        correctness_reward_func,
    ],
    args=training_args,
    train_dataset=dataset,
    # Removed custom callbacks to restore default table logging
)

# --- 5. RUN ---
print("Starting training...")
trainer.train()

INFO 11-24 01:36:01 [__init__.py:216] Automatically detected platform cuda.


The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting training...


wandb: Currently logged in as: hussenpremier03 (hussenpremier03-university-of-central-florida) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Initializing weave.
weave: Logged in as Weights & Biases user: hussenpremier03.
weave: View Weave data at https://wandb.ai/hussenpremier03-university-of-central-florida/huggingface/weave


Step,Training Loss
10,0.048100
20,-0.000400
30,0.029900
40,0.044000
50,0.034500
60,0.060100
70,0.060100
80,0.017600
90,0.020200
100,0.015100


TrainOutput(global_step=500, training_loss=0.017668356131762267, metrics={'train_runtime': 6847.917, 'train_samples_per_second': 4.673, 'train_steps_per_second': 0.073, 'total_flos': 0.0, 'train_loss': 0.017668356131762267})

# Evaluation

In [9]:
def evaluate_model(
    model_path: str,
    batch_size: int = 4,
    num_samples: int = None,
    save_results: bool = True,
):

    print("Loading model?")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
    ).to(device)

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    print("Loading GSM8K test data...")
    dataset = load_dataset("openai/gsm8k", "main", split="test")
    if num_samples:
        dataset = dataset.select(range(num_samples))

    total_samples = len(dataset)
    print(f"Total test samples: {total_samples}")

    results = []
    correct = 0
    total = 0

    progress_bar = tqdm(range(0, total_samples, batch_size), desc="Evaluating")


    # Batch Evaluation Loop
    for i in progress_bar:
        batch_data = dataset[i:i + batch_size]
        questions = batch_data["question"]

        # Format prompts exactly like training
        prompts = [
            [
                {"role": "system", "content": SYSTEM_PROMPT + "\n"},
                {"role": "user", "content": q.strip()}
            ]
            for q in questions
        ]

        # Convert chat prompts to strings using the chat template
        formatted_prompts = [
            tokenizer.apply_chat_template(
                p,
                tokenize=False,
                add_generation_prompt=True
            )
            for p in prompts
        ]

        # Tokenize batch
        inputs = tokenizer(
            formatted_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=768
        ).to(device)

        # Generate
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                pad_token_id=tokenizer.eos_token_id,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        # Process each response in the batch
        for j, response in enumerate(decoded):
            predicted = extract_xml_answer(response)
            truth = extract_hash_answer(batch_data["answer"][j])

            correct_flag = predicted == truth

            results.append({
                "question": questions[j],
                "true_answer": truth,
                "generated_answer": predicted,
                "full_response": response,
                "correct": correct_flag,
            })

            if correct_flag:
                correct += 1
            total += 1

        progress_bar.set_postfix({
            "accuracy": f"{(correct/total)*100:.2f}%",
            "correct": f"{correct}/{total}"
        })

    # Final Metrics
    accuracy = correct / total if total > 0 else 0
    metrics = {
        "accuracy": accuracy,
        "correct": correct,
        "total": total,
        "model_path": model_path,
        "timestamp": datetime.now().isoformat(),
    }


    # Save Results
    if save_results:
        save_path = f"gsm8k_eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(save_path, "w") as f:
            json.dump({"metrics": metrics, "results": results}, f, indent=2)
        print(f"\nResults saved to {save_path}")

    print("\nFinal Results")
    print(f"Accuracy: {accuracy:.2%}")
    print(f"Correct: {correct}/{total}")

    return metrics


Base model

In [10]:
evaluate_model(
    model_path="Qwen/Qwen2.5-0.5B-Instruct",
    batch_size=16,
    save_results=True,
)

Loading model?
Loading GSM8K test data...
Total test samples: 1319


Evaluating: 100%|██████████| 83/83 [19:00<00:00, 13.74s/it, accuracy=0.00%, correct=0/1319]


Results saved to gsm8k_eval_20251124_042038.json

Final Results
Accuracy: 0.00%
Correct: 0/1319


{'accuracy': 0.0,
 'correct': 0,
 'total': 1319,
 'model_path': 'Qwen/Qwen2.5-0.5B-Instruct',
 'timestamp': '2025-11-24T04:20:38.515404'}

Traind model with correct reward functions

In [11]:
evaluate_model(
    model_path="outputs/checkpoint-500",
    batch_size=16,
    save_results=True,
)

Loading model?
Loading GSM8K test data...
Total test samples: 1319


Evaluating: 100%|██████████| 83/83 [12:37<00:00,  9.13s/it, accuracy=44.88%, correct=592/1319]


Results saved to gsm8k_eval_20251124_043330.json

Final Results
Accuracy: 44.88%
Correct: 592/1319


{'accuracy': 0.4488248673237301,
 'correct': 592,
 'total': 1319,
 'model_path': 'outputs/checkpoint-500',
 'timestamp': '2025-11-24T04:33:30.557084'}

Trained initial model

In [12]:
evaluate_model(
    model_path="Qwen2-0.5B-Instruct-gsm8k-grpo-run1",
    batch_size=16,
    save_results=True,
)

Loading model?
Loading GSM8K test data...
Total test samples: 1319


Evaluating: 100%|██████████| 83/83 [15:55<00:00, 11.52s/it, accuracy=28.13%, correct=371/1319]


Results saved to gsm8k_eval_20251124_050438.json

Final Results
Accuracy: 28.13%
Correct: 371/1319


{'accuracy': 0.2812736921910538,
 'correct': 371,
 'total': 1319,
 'model_path': 'Qwen2-0.5B-Instruct-gsm8k-grpo-run1',
 'timestamp': '2025-11-24T05:04:38.821913'}

Sample prompt and answer

In [3]:
def ask_model(
    model_path: str,
    question: str,
    max_new_tokens: int = 300
):
    print(f"Loading model from {model_path} ...")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
    ).to(device)

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    # Build a single prompt like in training/evaluation
    chat_prompt = [
        {"role": "system", "content": SYSTEM_PROMPT + "\n"},
        {"role": "user", "content": question.strip()}
    ]

    # Convert to final formatted text (same as evaluation)
    formatted_prompt = tokenizer.apply_chat_template(
        chat_prompt,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=768,
    ).to(device)

    print("\nGenerating response...\n")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("===== MODEL OUTPUT =====")
    print(decoded)
    print("========================")

    return decoded


In [6]:
# Answer: 60
question= f"In a dance class of 20 students, 20% enrolled in contemporary dance, 25% of the remaining enrolled in jazz dance, and the rest enrolled in hip-hop dance. What percentage of the entire students enrolled in hip-hop dance?"

ask_model(
    model_path="qwen/Qwen2.5-0.5B-Instruct",
    question=question,
    max_new_tokens=300,
)

ask_model(
    model_path="outputs/checkpoint-500",
    question=question,
    max_new_tokens=300,
)

Loading model from qwen/Qwen2.5-0.5B-Instruct ...

Generating response...

===== MODEL OUTPUT =====
system

Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>


user
In a dance class of 20 students, 20% enrolled in contemporary dance, 25% of the remaining enrolled in jazz dance, and the rest enrolled in hip-hop dance. What percentage of the entire students enrolled in hip-hop dance?
assistant
To determine the percentage of the entire dance class that enrolled in hip-hop dance, we will follow these steps:

1. **Calculate the number of students enrolled in each category:**

   - Total students: 20
   - Percentage enrolled in contemporary dance: \(20\%\)
     \[
     \text{Contemporary Dance Students} = 20 \times \frac{20}{100} = 4
     \]
   - Remaining students after those who are enrolled in contemporary dance: 
     \[
     20 - 4 = 16
     \]

   - Percentage enrolled in jazz dance: \(25\%\)
     \[
     \text{Jazz Dance Students} = 25\% \times 16 = 

'system\n\nRespond in the following format:\n<reasoning>\n...\n</reasoning>\n<answer>\n...\n</answer>\n\n\nuser\nIn a dance class of 20 students, 20% enrolled in contemporary dance, 25% of the remaining enrolled in jazz dance, and the rest enrolled in hip-hop dance. What percentage of the entire students enrolled in hip-hop dance?\nassistant\n<reasoning>\nThere are 20 students in total.\n20% enrolled in contemporary dance, so there are 20 * 0.20 = 4 students in contemporary dance.\nThis leaves us with 20 - 4 = 16 students.\n25% of these remaining students enrolled in jazz dance, so there are 16 * 0.25 = 4 students in jazz dance.\nThe rest, which is 16 - 4 = 12 students, are enrolled in hip-hop dance.\nTo find the percentage of the entire students enrolled in hip-hop dance, we divide 12 by 20 and multiply by 100: (12 / 20) * 100 = 60%\nTherefore, 60% of the entire students enrolled in hip-hop dance.\n</reasoning>\n<answer>\n60\n</answer>\n'